# 02 — Garment swap + quality evaluation

**Phase-0, Stage B** of [03-generation-pipeline](../../docs/architecture/03-generation-pipeline.md#evaluation).
For every garment in the eval set and every angle: segment the flat-lay, inpaint the house model's
clothing mask using the flat-lay as **reference image**, then paste back everything outside the mask, score
the candidates, and let you mark each pass/fail. The result is a report with the **exit verdict**:
≥ 70% pass on *simple upper-body garments, front + 3/4 views*.

**Inputs**
- House model folder from notebook 01 (has `meta.json`, `poses/<angle>/base.png`, `mask_*.png`).
- Eval photos + `garments_manifest.csv` (see [eval/README](../eval/README.md)). On Kaggle, upload them as a private Dataset and add it as input.

Same Kaggle/Colab setup as notebook 01 (GPU T4, Internet on, `HF_TOKEN` secret).

## 1. Config

In [ ]:
import os
KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

MODEL_ID = "black-forest-labs/FLUX.2-klein-4B"
EMBED_MODEL_ID = "facebook/dinov2-small"          # Apache 2.0, used for pattern similarity
ENGINE_TAG = "klein4b-inpaint-v0"                 # goes into every result row

HOUSE_MODEL_DIR = None    # None = auto-find a folder with meta.json under /kaggle/input, OUT_DIR, or ./outputs
EVAL_DIR = None           # None = auto-find a folder containing garments_manifest.csv (+ the photos)
OUT_DIR = "/kaggle/working/eval_runs" if KAGGLE else "eval_runs"

ANGLES_TO_RUN = ["front", "three_quarter_left", "three_quarter_right", "back"]   # add "side" (beta) if you like
ONLY_GARMENTS = None      # e.g. ["g01", "g02"] for a quick first run
N_CANDIDATES = 4
SEEDS = [11, 22, 33, 44][:N_CANDIDATES]
STEPS = 4
GUIDANCE = 1.0
STRENGTH = 1.0            # 1.0 = ignore the base outfit pixels inside the mask
DILATE_PX = 14            # grow the mask so a slightly bigger garment fits
FEATHER_PX = 10           # soft edge, then hard paste-back outside it
FORCE_FP16 = False

## 2. Setup

In [ ]:
# --- Get the repo helpers + install pinned deps (runs once per session) ---
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/vishwa9/tryon-app.git"
REPO_BRANCH = "main"   # use "research/klein-house-model" until that branch is merged

def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents, Path("/kaggle/working/tryon-app"), Path("/content/tryon-app")]:
        if (p / "research" / "phase0_utils.py").exists():
            return p
    return None

REPO = _find_repo()
if REPO is None:
    REPO = Path("/kaggle/working/tryon-app" if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") else "/content/tryon-app")
    url = REPO_URL
    # Private repo? Add a GITHUB_TOKEN secret (read-only, fine-grained) in Kaggle/Colab secrets.
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        try:
            from google.colab import userdata
            tok = userdata.get("GITHUB_TOKEN")
        except Exception:
            tok = os.environ.get("GITHUB_TOKEN")
    if tok:
        url = url.replace("https://", f"https://x-access-token:{tok}@")
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", REPO_BRANCH, url, str(REPO)], check=True)
    if not (REPO / "research" / "phase0_utils.py").exists():
        raise SystemExit(f"Branch {REPO_BRANCH!r} has no research/phase0_utils.py. Set REPO_BRANCH to the branch with the notebooks.")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "research" / "requirements.txt")], check=True)
sys.path.insert(0, str(REPO / "research"))

import importlib, phase0_utils as u
importlib.reload(u)
print("repo:", REPO, "| runtime:", u.detect_runtime())

In [ ]:
# --- GPU check + Hugging Face login ---
import torch, diffusers
from huggingface_hub import login

if not torch.cuda.is_available():
    raise SystemExit("No GPU. Kaggle: Settings → Accelerator → GPU T4. Colab: Runtime → Change runtime type → T4 GPU.")
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name}, {props.total_memory/1e9:.1f} GB, compute {props.major}.{props.minor}")
print("torch", torch.__version__, "| diffusers", diffusers.__version__)

HF_TOKEN = u.get_secret("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
else:
    print("No HF_TOKEN secret found. Public models still download, but add one if you hit rate limits or gated repos.")

# T4/P100 (compute < 8.0) have no native bf16: it still works but is slower.
# float16 is faster on those cards but FLUX-family models can overflow to black/NaN images in fp16.
DTYPE = torch.bfloat16 if (props.major >= 8 or not FORCE_FP16) else torch.float16
print("dtype:", DTYPE)

def peak_vram_gb():
    return round(torch.cuda.max_memory_allocated() / 1e9, 2)

In [ ]:
# --- Locate inputs ---
from pathlib import Path
import json

def _search_roots():
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd(), REPO / "research"]
    return [r for r in roots if r.exists()]

def find_dir(marker, explicit):
    if explicit:
        return Path(explicit)
    for root in _search_roots():
        hits = sorted(root.rglob(marker))
        hits = [h for h in hits if ".ipynb_checkpoints" not in str(h)]
        if marker == "meta.json":
            hits = [h for h in hits if (h.parent / "poses").is_dir()]
        if marker == "garments_manifest.csv":
            # prefer a manifest whose folder actually has photos next to it
            hits.sort(key=lambda h: -sum(1 for _ in h.parent.glob("*.jp*g")) - sum(1 for _ in h.parent.glob("*.png")))
        if hits:
            return hits[0].parent
    raise FileNotFoundError(f"Couldn't find {marker}; set it in the config cell.")

HOUSE_MODEL_DIR = find_dir("meta.json", HOUSE_MODEL_DIR)
EVAL_DIR = find_dir("garments_manifest.csv", EVAL_DIR)
hm_meta = json.loads((HOUSE_MODEL_DIR / "meta.json").read_text())
print("house model:", hm_meta["name"], "@", HOUSE_MODEL_DIR)
print("eval dir:", EVAL_DIR)

## 3. Load models

In [ ]:
from diffusers import Flux2KleinInpaintPipeline
from transformers import AutoImageProcessor, AutoModel

timings = {}
with u.timer(timings, "load_s"):
    pipe = Flux2KleinInpaintPipeline.from_pretrained(MODEL_ID, torch_dtype=DTYPE)
    pipe.enable_model_cpu_offload()
emb_proc = AutoImageProcessor.from_pretrained(EMBED_MODEL_ID)
emb_model = AutoModel.from_pretrained(EMBED_MODEL_ID).to("cuda").eval()
print(timings)

def embed(img):
    with torch.no_grad():
        x = emb_proc(images=img.convert("RGB"), return_tensors="pt").to("cuda")
        return emb_model(**x).pooler_output[0].float().cpu().numpy()

## 4. House-model bases and masks

In [ ]:
import numpy as np
from PIL import Image

bases, hard_masks = {}, {}
for angle in ANGLES_TO_RUN:
    d = HOUSE_MODEL_DIR / "poses" / angle
    if not (d / "base.png").exists():
        print(f"⚠ no base for {angle}; it will be skipped"); continue
    bases[angle] = Image.open(d / "base.png").convert("RGB")
    hard_masks[angle] = {p.stem.removeprefix("mask_"): np.asarray(Image.open(p).convert("L")) > 127
                         for p in d.glob("mask_*.png")}
W, H = next(iter(bases.values())).size
assert W % 16 == 0 and H % 16 == 0, (W, H)
u.make_grid(list(bases.values()), list(bases.keys()), cols=len(bases), cell=240)

## 5. Garments: load, check, prepare

Each flat-lay is checked (blurry? garment cut off? background not plain?), then its background is removed,
it is white-balanced against the backdrop, and cropped. Rows that fail checks are listed and skipped. Retake those photos.

In [ ]:
import pandas as pd
manifest = pd.read_csv(EVAL_DIR / "garments_manifest.csv", dtype=str).fillna("")
if ONLY_GARMENTS:
    manifest = manifest[manifest.garment_id.isin(ONLY_GARMENTS)]

garments, checks = {}, []
for r in manifest.itertuples():
    entry = {"row": r}
    for side, fname in (("front", r.front_file), ("back", r.back_file)):
        if not fname:
            continue
        p = EVAL_DIR / fname
        if not p.exists():
            checks.append({"garment_id": r.garment_id, "side": side, "ok": False, "reasons": "FILE_MISSING"}); continue
        crop, cmask, chk = u.prepare_garment(u.load_rgb(p))
        checks.append({"garment_id": r.garment_id, "side": side, "ok": chk.ok, "reasons": ",".join(chk.reasons),
                       "blur": chk.blur, "coverage": chk.coverage})
        if chk.ok:
            entry[side] = (crop, cmask)
    if "front" in entry:
        garments[r.garment_id] = entry
checks_df = pd.DataFrame(checks)
display(checks_df[~checks_df.ok] if len(checks_df) else checks_df)
print(f"{len(garments)} garments ready")
u.make_grid([g["front"][0] for g in garments.values()], list(garments.keys()), cols=5, cell=200)

## 6. Swap + score

Resumable: results are appended to `scores.csv` after every garment, and finished (garment, angle) pairs
are skipped if you re-run after a disconnect.

In [ ]:
run_dir = Path(OUT_DIR) / f"{ENGINE_TAG}_{hm_meta['name']}"
run_dir.mkdir(parents=True, exist_ok=True)
scores_path = run_dir / "scores.csv"
scores = u.normalize_scores(pd.read_csv(scores_path) if scores_path.exists() else pd.DataFrame(columns=u.SCORE_COLUMNS))
done = set(zip(scores.garment_id, scores.angle))
skipped = []

def mask_for(row, angle):
    region = u.CATEGORY_REGION[row.category]
    key = "upper_arms" if (region == "upper" and row.include_arms.lower() in ("yes", "y", "true", "1")) else region
    return hard_masks[angle][key]

for gid, g in garments.items():
    row = g["row"]
    new_rows = []
    for angle in bases:
        if (gid, angle) in done:
            continue
        which, skip = u.garment_ref_for_angle(angle, has_back="back" in g)
        if skip:
            skipped.append((gid, angle, skip)); continue
        ref_img, ref_mask = g[which]
        hard = mask_for(row, angle)
        if hard.mean() < 0.01:
            skipped.append((gid, angle, "EMPTY_BASE_MASK")); continue
        alpha = u.dilate_and_feather(hard, DILATE_PX, FEATHER_PX)
        mask_img = u.mask_to_pil(alpha)
        ref_emb = embed(ref_img)
        out_dir = run_dir / gid / angle
        out_dir.mkdir(parents=True, exist_ok=True)
        for ci, seed in enumerate(SEEDS):
            torch.cuda.reset_peak_memory_stats()
            t = {}
            with u.timer(t, "s"):
                gen = pipe(prompt=u.swap_prompt(row.category, angle), image=bases[angle], mask_image=mask_img,
                           image_reference=ref_img, strength=STRENGTH, width=W, height=H,
                           num_inference_steps=STEPS, guidance_scale=GUIDANCE,
                           generator=torch.Generator("cpu").manual_seed(seed)).images[0]
            out = u.composite(bases[angle], gen, alpha)
            out.save(out_dir / f"{ci}.png")
            # score on the hard garment area; pattern on a crop with the rest painted white (like the flat-lay)
            dE = u.colour_delta_e(ref_img, ref_mask, out, hard)
            box = u.crop_box(hard)
            crop = np.asarray(out).copy(); crop[~hard] = 255
            psim = u.cosine(ref_emb, embed(Image.fromarray(crop).crop(box)))
            leak = u.leak_score(bases[angle], out, alpha)
            new_rows.append({"garment_id": gid, "category": row.category, "difficulty": row.difficulty,
                             "angle": angle, "candidate": ci, "seed": seed,
                             "colour_dE": round(dE, 2), "pattern_sim": round(psim, 4), "leak": round(leak, 5),
                             "total": u.total_score(dE, psim, leak), "pass": "", "fail_reason": "",
                             "sec_per_img": t["s"], "peak_vram_gb": peak_vram_gb(), "engine": ENGINE_TAG})
        print(f"{gid} {angle}: best total {max(r['total'] for r in new_rows if r['angle'] == angle):.3f}")
    if new_rows:
        scores = u.normalize_scores(pd.concat([scores, pd.DataFrame(new_rows)], ignore_index=True))
        scores.to_csv(scores_path, index=False)
print("skipped:", skipped)
print(scores.groupby("angle")["total"].describe()[["count", "mean", "min", "max"]] if len(scores) else "no results")

## 7. Review

For each garment × angle you see: **base | flat-lay | candidates sorted by score**.
Judge against the [quality bar](../../docs/architecture/00-overview.md#quality-bar-catalog-quality):
exact colour and print, right neckline/sleeves/buttons, no hand or edge artefacts, same person, studio look.

Then mark it with `mark(...)`. You only need to mark the candidate you'd ship, plus a rejected one with a reason if none is good.
Fail reasons: `COLOUR_MISMATCH`, `PRINT_OR_LOGO_WRONG`, `SHAPE_WRONG`, `ARTEFACT_HANDS`, `ARTEFACT_EDGES`, `IDENTITY_DRIFT`, `NOT_STUDIO_LOOK`, `OTHER`.

In [ ]:
def show(gid, angle):
    rows = scores[(scores.garment_id == gid) & (scores.angle == angle)].sort_values("total", ascending=False)
    which, _ = u.garment_ref_for_angle(angle, has_back="back" in garments[gid])
    imgs = [bases[angle], garments[gid][which][0]] + [Image.open(run_dir / gid / angle / f"{c}.png") for c in rows.candidate]
    labels = ["base", f"flat-lay ({which})"] + [f"c{c} total {t:.2f} dE {d:.1f}" for c, t, d in zip(rows.candidate, rows.total, rows.colour_dE)]
    display(u.make_grid(imgs, labels, cols=len(imgs), cell=300))

def mark(gid, angle, candidate, passed, reason=""):
    global scores
    if not passed and reason not in u.FAIL_REASONS:
        raise ValueError(f"reason must be one of {u.FAIL_REASONS}")
    sel = (scores.garment_id == gid) & (scores.angle == angle) & (scores.candidate == candidate)
    assert sel.sum() == 1, "no such result"
    scores.loc[sel, "pass"] = "Y" if passed else "N"
    scores.loc[sel, "fail_reason"] = "" if passed else reason
    scores.to_csv(scores_path, index=False)

pairs = scores[["garment_id", "angle"]].drop_duplicates().values.tolist()
print(len(pairs), "to review. Call show(gid, angle), then mark(...). Example below.")
if pairs:
    show(*pairs[0])

In [ ]:
# mark("g01", "front", candidate=2, passed=True)
# mark("g01", "three_quarter_left", candidate=0, passed=False, reason="PRINT_OR_LOGO_WRONG")

## 8. Report and exit verdict

In [ ]:
import datetime
summary = u.summarize(scores)
summary.update({"engine": ENGINE_TAG, "model_id": MODEL_ID, "house_model": hm_meta["name"],
                "gpu": torch.cuda.get_device_name(0), "dtype": str(DTYPE), "steps": STEPS,
                "n_candidates": N_CANDIDATES, "dilate_px": DILATE_PX, "feather_px": FEATHER_PX,
                "skipped": skipped, "date": datetime.date.today().isoformat()})
stem = f"{summary['date']}_{ENGINE_TAG}"
scores.to_csv(run_dir / f"{stem}.csv", index=False)
u.save_json(summary, run_dir / f"{stem}.json")
print(json.dumps({k: summary[k] for k in ("verdict", "exit_gate_pass_rate", "exit_gate_items", "reviewed",
                                          "pass_rate_by_angle", "top_fail_reasons", "median_sec_per_img")}, indent=2))
print(f"\nCommit {stem}.csv and {stem}.json to research/eval/reports/ (not the images).")